In [1]:
# ============================================================
# 00 — CONNECT GOOGLE DRIVE / PROJECT
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# ============================================================
# PROJECT PATH
# ============================================================

from pathlib import Path
import sys

PROJECT_DIR = Path(
    "/content/drive/MyDrive/FitnessML_Master"
)

print(f"Project directory: {PROJECT_DIR}")
print(f"Exists: {PROJECT_DIR.exists()}")

Project directory: /content/drive/MyDrive/FitnessML_Master
Exists: True


In [3]:
# ============================================================
# ADD PROJECT TO PYTHON PATH
# ============================================================

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Project added to Python path.")

Project added to Python path.


In [4]:
# ============================================================
# 003 — FITBIT PREPROCESSING
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np

import config_fitbit as config

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

RAW_DIR = Path(config.RAW_DATA_DIR)

print("=" * 70)
print("FITBIT PREPROCESSING")
print("=" * 70)

daily_activity = pd.read_csv(
    RAW_DIR / "dailyActivity_merged.csv"
)

daily_activity["ActivityDate"] = pd.to_datetime(
    daily_activity["ActivityDate"],
    format="%m/%d/%Y"
)

print(f"Raw shape: {daily_activity.shape}")

FITBIT PREPROCESSING
Raw shape: (940, 15)


In [5]:
# ============================================================
# DUPLICATES
# ============================================================

print("=" * 70)
print("DUPLICATE CHECK")
print("=" * 70)

duplicates = daily_activity.duplicated(
    subset=["Id", "ActivityDate"]
).sum()

print(f"Duplicate user/day records: {duplicates}")

DUPLICATE CHECK
Duplicate user/day records: 0


In [6]:
# ============================================================
# TEMPORAL ORDER
# ============================================================

daily_activity = (
    daily_activity
    .sort_values(["Id", "ActivityDate"])
    .reset_index(drop=True)
)

print("Sorted by user and date.")

display(
    daily_activity[
        ["Id", "ActivityDate", "TotalSteps", "Calories"]
    ].head(15)
)

Sorted by user and date.


,Id,ActivityDate,TotalSteps,Calories
0,1503960366,2016-04-12,13162,1985
1,1503960366,2016-04-13,10735,1797
2,1503960366,2016-04-14,10460,1776
3,1503960366,2016-04-15,9762,1745
4,1503960366,2016-04-16,12669,1863
5,1503960366,2016-04-17,9705,1728
6,1503960366,2016-04-18,13019,1921
7,1503960366,2016-04-19,15506,2035
8,1503960366,2016-04-20,10544,1786
9,1503960366,2016-04-21,9819,1775


In [7]:
# ============================================================
# MODELING COLUMNS
# ============================================================

feature_columns = [
    "TotalSteps",
    "TotalDistance",
    "TrackerDistance",
    "LoggedActivitiesDistance",
    "VeryActiveDistance",
    "ModeratelyActiveDistance",
    "LightActiveDistance",
    "SedentaryActiveDistance",
    "VeryActiveMinutes",
    "FairlyActiveMinutes",
    "LightlyActiveMinutes",
    "SedentaryMinutes"
]

target_column = "Calories"

print("=" * 70)
print("MODELING SETUP")
print("=" * 70)

print(f"Features: {len(feature_columns)}")
print(f"Target: {target_column}")

MODELING SETUP
Features: 12
Target: Calories


In [8]:
# ============================================================
# MISSING VALUES
# ============================================================

print("=" * 70)
print("MISSING VALUES — MODELING COLUMNS")
print("=" * 70)

missing = daily_activity[
    feature_columns + [target_column]
].isna().sum()

missing_pct = (
    missing / len(daily_activity) * 100
)

missing_table = pd.DataFrame({
    "missing": missing,
    "missing_%": missing_pct
})

display(
    missing_table[
        missing_table["missing"] > 0
    ]
)

MISSING VALUES — MODELING COLUMNS


,missing,missing_%


In [9]:
# ============================================================
# ZERO VALUES
# ============================================================

print("=" * 70)
print("ZERO VALUES — MODELING COLUMNS")
print("=" * 70)

zero_counts = (
    daily_activity[
        feature_columns + [target_column]
    ] == 0
).sum()

zero_pct = (
    zero_counts / len(daily_activity) * 100
)

zero_table = pd.DataFrame({
    "zeros": zero_counts,
    "zeros_%": zero_pct
})

display(
    zero_table[
        zero_table["zeros"] > 0
    ]
    .sort_values("zeros", ascending=False)
)

ZERO VALUES — MODELING COLUMNS


,zeros,zeros_%
LoggedActivitiesDistance,908,96.595745
SedentaryActiveDistance,858,91.276596
VeryActiveDistance,413,43.936170
VeryActiveMinutes,409,43.510638
ModeratelyActiveDistance,386,41.063830
FairlyActiveMinutes,384,40.851064
LightActiveDistance,85,9.042553
LightlyActiveMinutes,84,8.936170
TrackerDistance,78,8.297872
TotalDistance,78,8.297872


In [10]:
# ============================================================
# ZERO TARGET INSPECTION
# ============================================================

print("=" * 70)
print("ZERO CALORIES — INSPECTION")
print("=" * 70)

zero_calories = daily_activity[
    daily_activity["Calories"] == 0
][[
    "Id",
    "ActivityDate",
    "TotalSteps",
    "TotalDistance",
    "SedentaryMinutes",
    "Calories"
]]

display(zero_calories)

ZERO CALORIES — INSPECTION


,Id,ActivityDate,TotalSteps,TotalDistance,SedentaryMinutes,Calories
30,1503960366,2016-05-12,0,0.0,1440,0
653,6290855005,2016-05-10,0,0.0,1440,0
817,8253242879,2016-04-30,0,0.0,1440,0
879,8583815059,2016-05-12,0,0.0,1440,0


In [11]:
# ============================================================
# RANGE CHECK
# ============================================================

print("=" * 70)
print("RANGE CHECK — MODELING VARIABLES")
print("=" * 70)

range_table = daily_activity[
    feature_columns + [target_column]
].describe().T[
    ["min", "25%", "50%", "75%", "max"]
]

display(range_table)

RANGE CHECK — MODELING VARIABLES


,min,25%,50%,75%,max
TotalSteps,0.0,3789.750,7405.500,10727.0000,36019.000000
TotalDistance,0.0,2.620,5.245,7.7125,28.030001
TrackerDistance,0.0,2.620,5.245,7.7100,28.030001
LoggedActivitiesDistance,0.0,0.000,0.000,0.0000,4.942142
VeryActiveDistance,0.0,0.000,0.210,2.0525,21.920000
ModeratelyActiveDistance,0.0,0.000,0.240,0.8000,6.480000
LightActiveDistance,0.0,1.945,3.365,4.7825,10.710000
SedentaryActiveDistance,0.0,0.000,0.000,0.0000,0.110000
VeryActiveMinutes,0.0,0.000,4.000,32.0000,210.000000
FairlyActiveMinutes,0.0,0.000,6.000,19.0000,143.000000


In [12]:
# ============================================================
# NEGATIVE VALUES
# ============================================================

print("=" * 70)
print("NEGATIVE VALUES")
print("=" * 70)

negative_counts = (
    daily_activity[
        feature_columns + [target_column]
    ] < 0
).sum()

display(
    negative_counts[
        negative_counts > 0
    ]
)

NEGATIVE VALUES


,0


In [13]:
# ============================================================
# FINAL PREPROCESSED DATASET
# ============================================================

print("=" * 70)
print("FINAL PREPROCESSED DATASET")
print("=" * 70)

model_columns = [
    "Id",
    "ActivityDate"
] + feature_columns + [target_column]

daily_processed = (
    daily_activity[model_columns]
    .copy()
    .sort_values(["Id", "ActivityDate"])
    .reset_index(drop=True)
)

print(f"Rows: {len(daily_processed):,}")
print(f"Columns: {len(daily_processed.columns)}")
print(f"Users: {daily_processed['Id'].nunique()}")
print(
    f"Date range: "
    f"{daily_processed['ActivityDate'].min().date()} → "
    f"{daily_processed['ActivityDate'].max().date()}"
)

print()
print("Missing values:")
print(
    daily_processed.isna().sum().sum()
)

display(daily_processed.head())

FINAL PREPROCESSED DATASET
Rows: 940
Columns: 15
Users: 33
Date range: 2016-04-12 → 2016-05-12

Missing values:
0


,Id,ActivityDate,TotalSteps,TotalDistance,TrackerDistance,LoggedActivitiesDistance,VeryActiveDistance,ModeratelyActiveDistance,LightActiveDistance,SedentaryActiveDistance,VeryActiveMinutes,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,Calories
0,1503960366,2016-04-12,13162,8.50,8.50,0.0,1.88,0.55,6.06,0.0,25,13,328,728,1985
1,1503960366,2016-04-13,10735,6.97,6.97,0.0,1.57,0.69,4.71,0.0,21,19,217,776,1797
2,1503960366,2016-04-14,10460,6.74,6.74,0.0,2.44,0.40,3.91,0.0,30,11,181,1218,1776
3,1503960366,2016-04-15,9762,6.28,6.28,0.0,2.14,1.26,2.83,0.0,29,34,209,726,1745
4,1503960366,2016-04-16,12669,8.16,8.16,0.0,2.71,0.41,5.04,0.0,36,10,221,773,1863


In [16]:
# ============================================================
# SAVE PROCESSED DATASET
# ============================================================

raw_dir = Path(config.RAW_DATA_DIR)

processed_dir = raw_dir.parents[1] / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "fitbit_daily_processed.csv"

daily_processed.to_csv(
    output_path,
    index=False
)

print("=" * 70)
print("PROCESSED DATASET SAVED")
print("=" * 70)

print(f"Path: {output_path}")
print(f"Rows: {len(daily_processed):,}")
print(f"Columns: {len(daily_processed.columns)}")

PROCESSED DATASET SAVED
Path: /content/drive/MyDrive/FitnessML_Master/data/processed/fitbit_daily_processed.csv
Rows: 940
Columns: 15
